In [ ]:
# ============================================================
# Week 12 BBO Capstone Script
# Theme: PCA-Based Structure Learning
# Uses 21 data points
# GP + SVM + MLP + CNN + KMeans + PCA Guidance
# ============================================================

import numpy as np
import pandas as pd
import warnings
import re

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances
from sklearn.decomposition import PCA

import torch
import torch.nn as nn
import torch.optim as optim

warnings.filterwarnings("ignore")
np.random.seed(42)
torch.manual_seed(42)

# ============================================================
# Load historical inputs / outputs from uploaded text files
# ============================================================

def extract_python_blocks(text):
    blocks = []
    depth = 0
    current = ""

    for ch in text:
        if ch == "[":
            if depth == 0:
                current = ""
            depth += 1

        if depth > 0:
            current += ch

        if ch == "]":
            depth -= 1
            if depth == 0:
                blocks.append(current.strip())

    return blocks


def load_input_history(path):
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    blocks = extract_python_blocks(text)

    history = []
    for block in blocks:
        row = eval(block, {"array": np.array, "np": np})
        row = [np.array(x, dtype=float) for x in row]
        history.append(row)

    return history


def load_output_history(path):
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    blocks = extract_python_blocks(text)

    history = []
    for block in blocks:
        row = eval(block, {"np": np})
        row = [float(x) for x in row]
        history.append(row)

    return history


historical_inputs = load_input_history("Week_11_inputs.txt")
historical_outputs = load_output_history("Week_11_outputs.txt")

print("Loaded historical input rounds:", len(historical_inputs))
print("Loaded historical output rounds:", len(historical_outputs))

# ============================================================
# Helper Functions
# ============================================================

def pair_exists(X, y, point, output):
    same_x = np.all(np.isclose(X, point.reshape(1, -1), atol=1e-10), axis=1)
    same_y = np.isclose(y, output, atol=1e-10)
    return np.any(same_x & same_y)


def normalize_score(values):
    values = np.array(values, dtype=float)
    std = np.std(values)

    if std < 1e-12:
        return np.zeros_like(values)

    return (values - np.mean(values)) / std


def adaptive_strategy_params(dim, latest_improved, plateau_detected):
    if dim <= 3:
        beta = 1.2 if latest_improved else 1.9
        noise = 0.025 if latest_improved else 0.050
        temperature = 0.75 if latest_improved else 1.15
    elif dim <= 5:
        beta = 1.6 if latest_improved else 2.5
        noise = 0.040 if latest_improved else 0.075
        temperature = 0.85 if latest_improved else 1.25
    else:
        beta = 2.1 if latest_improved else 3.1
        noise = 0.070 if latest_improved else 0.110
        temperature = 0.95 if latest_improved else 1.35

    if plateau_detected:
        beta *= 1.20
        noise *= 1.25
        temperature *= 1.15

    return beta, noise, temperature


def pca_reasoning(explained_variance, pca_distance, pca_alignment):
    retained = explained_variance * 100
    return (
        f"PCA retained approximately {retained:.2f}% of the search-space variance. "
        f"The selected query is guided by principal-direction alignment "
        f"({pca_alignment:.6f}) and PCA-space distance ({pca_distance:.6f}), "
        f"helping reduce randomness while preserving meaningful exploration."
    )


def cluster_reasoning(best_cluster, cluster_mean, boundary_distance, nearest_centroid_distance):
    return (
        f"Query targets cluster {best_cluster}, which has a mean output of {cluster_mean:.6f}. "
        f"The candidate is guided by centroid proximity ({nearest_centroid_distance:.6f}) "
        f"and boundary distance ({boundary_distance:.6f})."
    )

# ============================================================
# CNN Surrogate
# ============================================================

class CNN1DSurrogate(nn.Module):
    def __init__(self, input_dim, channels=16):
        super().__init__()

        self.conv1 = nn.Conv1d(1, channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(channels, channels * 2, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear((channels * 2) * input_dim, 32)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        return self.fc2(x).squeeze(-1)


def train_cnn(X, y, channels=16, epochs=650, lr=0.008):
    X_tensor = torch.tensor(X, dtype=torch.float32)

    y_mean = np.mean(y)
    y_std = np.std(y) if np.std(y) > 1e-12 else 1.0

    y_scaled = (y - y_mean) / y_std
    y_tensor = torch.tensor(y_scaled, dtype=torch.float32)

    model = CNN1DSurrogate(X.shape[1], channels=channels)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.MSELoss()

    for _ in range(epochs):
        optimizer.zero_grad()
        pred = model(X_tensor)
        loss = loss_fn(pred, y_tensor)
        loss.backward()
        optimizer.step()

    return model, y_mean, y_std


def cnn_predict(model, X, y_mean, y_std):
    model.eval()

    with torch.no_grad():
        pred_scaled = model(torch.tensor(X, dtype=torch.float32)).numpy()

    return pred_scaled * y_std + y_mean


def cnn_gradient(model, x):
    model.eval()

    x_tensor = torch.tensor(
        x.reshape(1, -1),
        dtype=torch.float32,
        requires_grad=True
    )

    pred = model(x_tensor)
    pred.backward()

    return x_tensor.grad.detach().numpy().ravel()


def cnn_feature_importance(model, X):
    grads = []

    for x in X:
        grads.append(np.abs(cnn_gradient(model, x)))

    importance = np.mean(grads, axis=0)

    if importance.sum() > 0:
        importance = importance / importance.sum()

    return importance

# ============================================================
# Main Loop
# ============================================================

results = []
importance_rows = []
cluster_rows = []
pca_rows = []
decision_rows = []

for func_id in range(1, 9):

    print("\n" + "=" * 60)
    print(f"Function {func_id} - Week 12")
    print("=" * 60)

    X = np.load(f"function{func_id}_inputs.npy")
    y = np.load(f"function{func_id}_outputs.npy")

    # --------------------------------------------------------
    # Append all historical rounds from Week_11 files
    # --------------------------------------------------------
    for r in range(len(historical_inputs)):
        point = np.array(historical_inputs[r][func_id - 1], dtype=float)
        output = float(historical_outputs[r][func_id - 1])

        if point.shape[0] != X.shape[1]:
            raise ValueError(f"Function {func_id}: dimension mismatch.")

        if not pair_exists(X, y, point, output):
            X = np.vstack([X, point])
            y = np.append(y, output)

    dim = X.shape[1]

    best_idx = np.argmax(y)
    best_input = X[best_idx]
    best_output = y[best_idx]

    latest_output = historical_outputs[-1][func_id - 1]
    previous_best = np.max(y[:-1])
    latest_improved = latest_output >= previous_best

    recent_outputs = y[-5:]
    plateau_detected = (
        np.max(recent_outputs) - np.min(recent_outputs)
    ) < 0.05 * (np.std(y) + 1e-8)

    beta, noise, temperature = adaptive_strategy_params(
        dim,
        latest_improved,
        plateau_detected
    )

    print("Dataset size:", X.shape)
    print("Best output:", best_output)
    print("Best input:", best_input)

    # ========================================================
    # PCA Layer
    # ========================================================

    scaler_pca = StandardScaler()
    X_scaled = scaler_pca.fit_transform(X)

    n_pca_components = min(2, dim, len(X) - 1)

    pca = PCA(n_components=n_pca_components, random_state=42)
    X_pca = pca.fit_transform(X_scaled)

    explained_variance = float(np.sum(pca.explained_variance_ratio_))

    # Weighted PCA direction based on performance
    y_weights = y - np.min(y)
    if np.sum(y_weights) < 1e-12:
        y_weights = np.ones_like(y)

    y_weights = y_weights / np.sum(y_weights)

    performance_centroid_pca = np.average(X_pca, axis=0, weights=y_weights)

    best_pca_point = X_pca[best_idx]

    principal_direction = performance_centroid_pca - np.mean(X_pca, axis=0)

    if np.linalg.norm(principal_direction) > 1e-12:
        principal_direction = principal_direction / np.linalg.norm(principal_direction)

    print("PCA explained variance:", explained_variance)

    # ========================================================
    # Clustering Layer
    # ========================================================

    n_clusters = min(4, max(2, len(X) // 5))

    kmeans = KMeans(
        n_clusters=n_clusters,
        random_state=42,
        n_init=10
    )

    cluster_ids = kmeans.fit_predict(X)
    centroids = kmeans.cluster_centers_

    cluster_scores = []
    cluster_sizes = []

    for c in range(n_clusters):
        idx = np.where(cluster_ids == c)[0]
        cluster_sizes.append(len(idx))
        cluster_scores.append(float(np.mean(y[idx])))

    best_cluster = int(np.argmax(cluster_scores))
    best_cluster_centroid = centroids[best_cluster]

    print("Best cluster:", best_cluster)
    print("Best cluster mean:", cluster_scores[best_cluster])

    # ========================================================
    # GP Model
    # ========================================================

    kernel = (
        ConstantKernel(1.0)
        * RBF(length_scale=np.ones(dim))
        + WhiteKernel(noise_level=1e-5)
    )

    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        alpha=1e-6,
        random_state=42
    )

    gp.fit(X, y)

    # ========================================================
    # SVM Model
    # ========================================================

    threshold = np.quantile(y, 0.70)
    labels = (y >= threshold).astype(int)

    if len(np.unique(labels)) > 1:
        svm = Pipeline([
            ("scaler", StandardScaler()),
            ("svc", SVC(
                kernel="rbf",
                probability=True,
                C=1.0,
                gamma="scale",
                random_state=42
            ))
        ])

        svm.fit(X, labels)
        use_svm = True
    else:
        svm = None
        use_svm = False

    # ========================================================
    # MLP Model
    # ========================================================

    mlp = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPRegressor(
            hidden_layer_sizes=(128, 64, 32),
            activation="relu",
            alpha=0.0005,
            learning_rate_init=0.001,
            max_iter=1200,
            random_state=42
        ))
    ])

    mlp.fit(X, y)

    # ========================================================
    # CNN Model
    # ========================================================

    if dim <= 3:
        cnn_channels, cnn_epochs, cnn_lr = 8, 550, 0.010
    elif dim <= 5:
        cnn_channels, cnn_epochs, cnn_lr = 16, 700, 0.008
    else:
        cnn_channels, cnn_epochs, cnn_lr = 24, 850, 0.006

    cnn_model, y_mean, y_std = train_cnn(
        X,
        y,
        channels=cnn_channels,
        epochs=cnn_epochs,
        lr=cnn_lr
    )

    cnn_importance = cnn_feature_importance(cnn_model, X)

    imp_row = {"Function": func_id}

    for j in range(dim):
        imp_row[f"x{j+1}_importance"] = cnn_importance[j]

    importance_rows.append(imp_row)

    # ========================================================
    # Candidate Generation
    # ========================================================

    if dim <= 3:
        n_global, n_local, n_cluster, n_pca = 9000, 3000, 2500, 2500
        grad_lr, grad_steps = 0.035, 30
    elif dim <= 5:
        n_global, n_local, n_cluster, n_pca = 12000, 3500, 3000, 3000
        grad_lr, grad_steps = 0.045, 35
    else:
        n_global, n_local, n_cluster, n_pca = 16000, 4000, 3500, 3500
        grad_lr, grad_steps = 0.055, 40

    global_candidates = np.random.uniform(0, 1, size=(n_global, dim))

    local_candidates = np.clip(
        best_input + np.random.normal(0, noise, size=(n_local, dim)),
        0,
        1
    )

    cluster_candidates = np.clip(
        best_cluster_centroid + np.random.normal(0, noise, size=(n_cluster, dim)),
        0,
        1
    )

    # PCA candidates:
    # Generate candidates in PCA space around the best point
    # and performance-weighted centroid, then inverse transform
    pca_candidates_latent = []

    for _ in range(n_pca):
        if np.random.rand() < 0.5:
            base = best_pca_point
        else:
            base = performance_centroid_pca

        step = np.random.normal(0, noise, size=n_pca_components)

        if np.linalg.norm(principal_direction) > 1e-12:
            step += 0.30 * noise * principal_direction

        new_latent = base + step
        pca_candidates_latent.append(new_latent)

    pca_candidates_latent = np.array(pca_candidates_latent)

    # Rebuild to full PCA dimensional space if needed
    reconstructed_scaled = pca.inverse_transform(pca_candidates_latent)
    pca_candidates = scaler_pca.inverse_transform(reconstructed_scaled)
    pca_candidates = np.clip(pca_candidates, 0, 1)

    # CNN gradient candidates
    top_k = min(7, len(y))
    top_indices = np.argsort(y)[-top_k:]
    gradient_candidates = []

    for idx in top_indices:
        x = X[idx].copy()

        for _ in range(grad_steps):
            grad = cnn_gradient(cnn_model, x)
            grad = grad * (0.5 + cnn_importance)

            norm = np.linalg.norm(grad)

            if norm > 1e-12:
                grad = grad / norm

            x = np.clip(x + grad_lr * grad, 0, 1)

        gradient_candidates.append(x)

    gradient_candidates = np.array(gradient_candidates)

    candidates = np.vstack([
        global_candidates,
        local_candidates,
        cluster_candidates,
        pca_candidates,
        gradient_candidates
    ])

    # ========================================================
    # Scoring
    # ========================================================

    mu, sigma = gp.predict(candidates, return_std=True)
    ucb = mu + beta * sigma

    mlp_pred = mlp.predict(candidates)
    cnn_pred = cnn_predict(cnn_model, candidates, y_mean, y_std)

    if use_svm:
        svm_prob = svm.predict_proba(candidates)[:, 1]
    else:
        svm_prob = np.ones(len(candidates))

    grad_strength = np.array([
        np.linalg.norm(cnn_gradient(cnn_model, c))
        for c in candidates
    ])

    # Attention-style score: closeness to recent points
    recent_points = X[-5:]

    attention_dist = np.array([
        np.mean([np.linalg.norm(c - rp) for rp in recent_points])
        for c in candidates
    ])

    attention_score = normalize_score(1.0 / (attention_dist + 1e-6))

    # Diversity bonus: distance from all previous points
    diversity_dist = np.array([
        np.min([np.linalg.norm(c - old) for old in X])
        for c in candidates
    ])

    diversity_bonus = normalize_score(diversity_dist)

    # Cluster proximity
    cluster_dist = pairwise_distances(
        candidates,
        best_cluster_centroid.reshape(1, -1)
    ).flatten()

    cluster_score = normalize_score(np.exp(-cluster_dist))

    # Boundary tightening cue
    all_centroid_distances = pairwise_distances(candidates, centroids)
    sorted_distances = np.sort(all_centroid_distances, axis=1)

    boundary_distance = sorted_distances[:, 1] - sorted_distances[:, 0]
    boundary_score = normalize_score(-boundary_distance)

    # PCA scores
    candidates_scaled = scaler_pca.transform(candidates)
    candidates_pca = pca.transform(candidates_scaled)

    pca_distance = np.linalg.norm(
        candidates_pca - performance_centroid_pca.reshape(1, -1),
        axis=1
    )

    pca_distance_score = normalize_score(-pca_distance)

    if np.linalg.norm(principal_direction) > 1e-12:
        centered_candidates_pca = candidates_pca - np.mean(X_pca, axis=0)

        pca_alignment = centered_candidates_pca @ principal_direction
    else:
        pca_alignment = np.zeros(len(candidates))

    pca_alignment_score = normalize_score(pca_alignment)

    ucb_s = normalize_score(ucb)
    mlp_s = normalize_score(mlp_pred)
    cnn_s = normalize_score(cnn_pred)
    grad_s = normalize_score(grad_strength)

    base_score = (
        0.20 * ucb_s
        + 0.14 * mlp_s
        + 0.14 * cnn_s
        + 0.08 * svm_prob
        + 0.08 * grad_s
        + 0.07 * attention_score
        + 0.07 * diversity_bonus
        + 0.06 * cluster_score
        + 0.03 * boundary_score
        + 0.08 * pca_distance_score
        + 0.05 * pca_alignment_score
    )

    final_score = base_score / max(temperature, 1e-8)

    best_candidate_idx = np.argmax(final_score)
    new_point = np.clip(candidates[best_candidate_idx], 0, 1)

    query = "-".join([f"{v:.6f}" for v in new_point])

    selected_cluster_distance = float(cluster_dist[best_candidate_idx])
    selected_boundary_distance = float(boundary_distance[best_candidate_idx])
    selected_pca_distance = float(pca_distance[best_candidate_idx])
    selected_pca_alignment = float(pca_alignment[best_candidate_idx])

    cluster_text = cluster_reasoning(
        best_cluster,
        cluster_scores[best_cluster],
        selected_boundary_distance,
        selected_cluster_distance
    )

    pca_text = pca_reasoning(
        explained_variance,
        selected_pca_distance,
        selected_pca_alignment
    )

    reasoning = cluster_text + " " + pca_text

    print("Week 12 Submission:", query)
    print("Reasoning:", reasoning)

    # ========================================================
    # Save Logs
    # ========================================================

    results.append({
        "Function": func_id,
        "Dimension": dim,
        "Best_Output": float(best_output),
        "Week12_Query": query
    })

    cluster_rows.append({
        "Function": func_id,
        "Number_of_Clusters": int(n_clusters),
        "Best_Cluster": int(best_cluster),
        "Best_Cluster_Mean_Output": float(cluster_scores[best_cluster]),
        "Best_Cluster_Size": int(cluster_sizes[best_cluster]),
        "Selected_Cluster_Distance": selected_cluster_distance,
        "Selected_Boundary_Distance": selected_boundary_distance
    })

    pca_rows.append({
        "Function": func_id,
        "PCA_Components": int(n_pca_components),
        "Explained_Variance_Ratio": float(explained_variance),
        "Selected_PCA_Distance": selected_pca_distance,
        "Selected_PCA_Alignment": selected_pca_alignment
    })

    decision_rows.append({
        "Function": func_id,
        "Dimension": dim,
        "Best_Output": float(best_output),
        "Latest_Output": float(latest_output),
        "Latest_Improved": bool(latest_improved),
        "Plateau_Detected": bool(plateau_detected),
        "Beta": float(beta),
        "Noise": float(noise),
        "Temperature": float(temperature),
        "Week12_Query": query,
        "Selected_UCB_Component": float(ucb_s[best_candidate_idx]),
        "Selected_MLP_Component": float(mlp_s[best_candidate_idx]),
        "Selected_CNN_Component": float(cnn_s[best_candidate_idx]),
        "Selected_SVM_Probability": float(svm_prob[best_candidate_idx]),
        "Selected_Gradient_Component": float(grad_s[best_candidate_idx]),
        "Selected_Attention_Component": float(attention_score[best_candidate_idx]),
        "Selected_Diversity_Component": float(diversity_bonus[best_candidate_idx]),
        "Selected_Cluster_Component": float(cluster_score[best_candidate_idx]),
        "Selected_Boundary_Component": float(boundary_score[best_candidate_idx]),
        "Selected_PCA_Distance_Component": float(pca_distance_score[best_candidate_idx]),
        "Selected_PCA_Alignment_Component": float(pca_alignment_score[best_candidate_idx]),
        "Reasoning": reasoning
    })

# ============================================================
# Save Results
# ============================================================

pd.DataFrame(results).to_csv("Week12_Submissions.csv", index=False)
pd.DataFrame(importance_rows).to_csv("Week12_CNN_Feature_Importance.csv", index=False)
pd.DataFrame(cluster_rows).to_csv("Week12_Cluster_Log.csv", index=False)
pd.DataFrame(pca_rows).to_csv("Week12_PCA_Log.csv", index=False)
pd.DataFrame(decision_rows).to_csv("Week12_Decision_Log.csv", index=False)

print("\nDone ✅")
print("Saved: Week12_Submissions.csv")
print("Saved: Week12_CNN_Feature_Importance.csv")
print("Saved: Week12_Cluster_Log.csv")
print("Saved: Week12_PCA_Log.csv")
print("Saved: Week12_Decision_Log.csv")

Loaded historical input rounds: 11
Loaded historical output rounds: 11

Function 1 - Week 12
Dataset size: (20, 2)
Best output: 7.710875114502849e-16
Best input: [0.73102363 0.73299988]
PCA explained variance: 1.0
Best cluster: 2
Best cluster mean: 3.1149282340843297e-48
